# WP1: CT-RATE Data Ingestion & Structural Review

The goal of this notebook is to load the CT-RATE dataset, understand its structure, and prepare a combined dataset for downstream analysis in WP2 and WP3.

CT-RATE is a large chest CT dataset paired with radiology reports. It consists of three parts:
- **reports**: free-text radiology reports (findings + impressions)
- **metadata**: technical scan parameters and patient demographics
- **labels**: automatically extracted abnormality labels

All three are linked via the `VolumeName` key.

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
import os

DATASET_NAME = "ibrahimhamamci/CT-RATE"
OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Setup complete.")

## 1. Loading the Dataset

The CT-RATE dataset is hosted on HuggingFace. We load all three subsets separately and will merge them later using the shared `VolumeName` key.

In [ ]:
print("Loading CT-RATE dataset from HuggingFace...")

metadata_dataset = load_dataset(DATASET_NAME, "metadata")
reports_dataset  = load_dataset(DATASET_NAME, "reports")
labels_dataset   = load_dataset(DATASET_NAME, "labels")

print(f"Reports:  {len(reports_dataset['train'])} train | {len(reports_dataset['validation'])} val")
print(f"Metadata: {len(metadata_dataset['train'])} train | {len(metadata_dataset['validation'])} val")
print(f"Labels:   {len(labels_dataset['train'])} train | {len(labels_dataset['validation'])} val")

## 2. Converting to DataFrames

Working with pandas DataFrames is much more convenient for exploration and analysis. Let's convert all three datasets and take a first look at the columns.

In [ ]:
df_reports  = pd.DataFrame(reports_dataset['train'])
df_metadata = pd.DataFrame(metadata_dataset['train'])
df_labels   = pd.DataFrame(labels_dataset['train'])

print("DataFrames created:")
print(f"  Reports:  {df_reports.shape}")
print(f"  Metadata: {df_metadata.shape}")
print(f"  Labels:   {df_labels.shape}")

In [ ]:
# Quick look at the columns in each DataFrame
print("Report columns:")
print(df_reports.columns.tolist())

print("\nMetadata columns:")
print(df_metadata.columns.tolist())

print("\nLabel columns (abnormalities):")
print(df_labels.columns.tolist())

## 3. Merging the Datasets

All three DataFrames share the `VolumeName` column which acts as a unique identifier for each CT volume. We use this to merge them into one combined DataFrame.

In [ ]:
df_combined = df_reports.merge(df_metadata, on='VolumeName', how='inner')
df_combined = df_combined.merge(df_labels,   on='VolumeName', how='inner')

print(f"Combined dataset: {df_combined.shape[0]} rows, {df_combined.shape[1]} columns")
print(f"\nFirst few rows:")
df_combined.head(3)

## 4. Structural Checks

Before doing any analysis, it's important to check the data quality: are there duplicates? Any missing values in the key columns we'll be working with?

In [ ]:
print("=== Structural Checks ===")
print(f"Duplicate VolumeNames: {df_combined['VolumeName'].duplicated().sum()}")

# Check missing values - only show columns that actually have missing data
nan_counts = df_combined.isnull().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)

print(f"\nMissing values:")
if nan_counts.empty:
    print("  None - dataset is complete!")
else:
    for col, count in nan_counts.head(10).items():
        pct = count / len(df_combined) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")

## 5. Exploratory Analysis

Let's get a better feel for the data before saving it. The CT-RATE paper mentions substantial heterogeneity in patient demographics, scan parameters, and report styles - let's verify this.

In [ ]:
# Report text lengths - important for WP2 style analysis
df_combined['findings_len']    = df_combined['Findings_EN'].fillna('').apply(lambda x: len(x.split()))
df_combined['impressions_len'] = df_combined['Impressions_EN'].fillna('').apply(lambda x: len(x.split()))

print("=== Report Length Statistics (words) ===")
print(f"\nFindings:")
print(f"  Mean:   {df_combined['findings_len'].mean():.1f}")
print(f"  Median: {df_combined['findings_len'].median():.1f}")
print(f"  Std:    {df_combined['findings_len'].std():.1f}")
print(f"  Min:    {df_combined['findings_len'].min()}")
print(f"  Max:    {df_combined['findings_len'].max()}")

print(f"\nImpressions:")
print(f"  Mean:   {df_combined['impressions_len'].mean():.1f}")
print(f"  Median: {df_combined['impressions_len'].median():.1f}")
print(f"  Std:    {df_combined['impressions_len'].std():.1f}")

In [ ]:
# Patient demographics
print("=== Patient Demographics ===")

print(f"\nSex distribution:")
print(df_combined['PatientSex'].value_counts())

# Clean up age column - it comes as strings like '049Y'
df_combined['PatientAge_num'] = pd.to_numeric(
    df_combined['PatientAge'].str.replace('Y', '').str.replace('y', ''),
    errors='coerce'
)

print(f"\nAge distribution:")
print(f"  Mean:   {df_combined['PatientAge_num'].mean():.1f} years")
print(f"  Median: {df_combined['PatientAge_num'].median():.1f} years")
print(f"  Range:  {df_combined['PatientAge_num'].min():.0f} - {df_combined['PatientAge_num'].max():.0f} years")

print(f"\nManufacturer distribution:")
print(df_combined['Manufacturer'].value_counts())

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CT-RATE Dataset Overview', fontsize=14, fontweight='bold')

# Plot 1: Report length distribution
ax = axes[0, 0]
ax.hist(df_combined['findings_len'], bins=50,
        color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df_combined['findings_len'].median(), color='red',
           linestyle='--', label=f"Median: {df_combined['findings_len'].median():.0f}")
ax.set_title('Findings Length Distribution (words)')
ax.set_xlabel('Number of words')
ax.set_ylabel('Count')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Plot 2: Age distribution
ax = axes[0, 1]
ax.hist(df_combined['PatientAge_num'].dropna(), bins=40,
        color='darkorange', edgecolor='white', alpha=0.8)
ax.set_title('Patient Age Distribution')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Plot 3: Manufacturer distribution
ax = axes[1, 0]
manufacturer_counts = df_combined['Manufacturer'].value_counts()
ax.bar(manufacturer_counts.index, manufacturer_counts.values,
       color='green', alpha=0.7, edgecolor='white')
ax.set_title('CT Scanner Manufacturer')
ax.set_xlabel('Manufacturer')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Plot 4: Sex distribution
ax = axes[1, 1]
sex_counts = df_combined['PatientSex'].value_counts()
ax.pie(sex_counts.values, labels=sex_counts.index,
       autopct='%1.1f%%', colors=['steelblue', 'darkorange', 'gray'])
ax.set_title('Patient Sex Distribution')

plt.tight_layout()
plt.savefig('wp1_dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: wp1_dataset_overview.png")

In [ ]:
# Look at a sample report to understand the structure
sample_idx = 7  # this index has clinical information filled in
sample = df_combined.iloc[sample_idx]

print(f"Sample Report (index {sample_idx}):")
print(f"VolumeName: {sample['VolumeName']}")
print(f"Patient: {sample['PatientSex']}, {sample['PatientAge']}")
print(f"Scanner: {sample['Manufacturer']} - {sample['ManufacturerModelName']}")
print()

for field in ['ClinicalInformation_EN', 'Technique_EN', 'Findings_EN', 'Impressions_EN']:
    text = str(sample[field])
    print(f"--- {field} ({len(text.split())} words) ---")
    print(textwrap.fill(text, width=80)[:300] + "...\n")

## 6. Key Fields for Downstream Analysis

Based on the exploration above, here is a summary of the fields that will be most relevant for WP2 (style analysis) and WP3 (normalization):

| Category | Fields |
|---|---|
| **Report Text** | `Findings_EN`, `Impressions_EN`, `ClinicalInformation_EN`, `Technique_EN` |
| **Patient Info** | `PatientSex`, `PatientAge` |
| **Scan Info** | `StudyDate`, `Manufacturer`, `SeriesDescription`, `CTDIvol`, `NumberofSlices` |
| **Pathology Labels** | 18 binary abnormality columns from the labels dataset |

The `VolumeName` column is the key that links everything together. Note that each patient can have multiple volumes (different reconstruction kernels), which needs to be handled in WP2.

## 7. Saving the Combined Dataset

In [ ]:
# Drop the helper columns we added for analysis before saving
df_save = df_combined.drop(columns=['findings_len', 'impressions_len', 'PatientAge_num'])

output_path = os.path.join(OUTPUT_DIR, "ctrate_train_combined.csv")
df_save.to_csv(output_path, index=False)

print(f"Dataset saved: {output_path}")
print(f"Shape: {df_save.shape}")
print(f"\nWP1 complete!")